# N087 · 二分图、增广路与匹配

**目标：** 用交替路径扩展匹配并识别适用图结构。

**先修：** N080, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 二染色；奇环；匹配；交替/增广路；Kőnig定理的条件。

## 从问题到算法

二分图的约束是每条边两端颜色相反。匹配则要求每个点至多参加一条选定边。遇到已占用的右点不能立即放弃：沿“未匹配边、匹配边”交替行走，若到达空闲右点，就翻转整条路径，把匹配数增加1。这里用显式队列和前驱重建，避免长增广链递归溢出。

## 最小实现

**本章接口：** `is_bipartite(adj)`、`maximum_bipartite_matching(left_adj, right_size)`

In [1]:
from collections import deque

def is_bipartite(adj):
    color=[-1]*len(adj)
    for s in range(len(adj)):
        if color[s]!=-1: continue
        color[s]=0; q=deque([s])
        while q:
            u=q.popleft()
            for v in adj[u]:
                if color[v]==color[u]: return False
                if color[v]==-1: color[v]=1-color[u]; q.append(v)
    return True

def maximum_bipartite_matching(left_adj,right_size):
    left=[-1]*len(left_adj); right=[-1]*right_size
    for start in range(len(left)):
        q=deque([start]); seen={start}; prev={}; end=-1
        while q and end==-1:
            u=q.popleft()
            for v in left_adj[u]:
                if v in prev: continue
                prev[v]=u
                if right[v]==-1: end=v; break
                nxt=right[v]
                if nxt not in seen: seen.add(nxt); q.append(nxt)
        while end!=-1:
            u=prev[end]; old=left[u]
            left[u]=end; right[end]=u; end=old
    return sum(v!=-1 for v in left),left

## 正确性、前提与复杂度

交替路径翻转后，内部点仍各配一条边，两个端点从空闲变为已配。若不存在增广路，却有更大匹配，两匹配的对称差必含一个增广路径，矛盾。因此逐次增广至不能增广即为最大匹配。染色时同色边正好违反二分约束。

**代价：** 染色O(V+E)；本实现每个左点一次O(V+E)搜索，总O(L(V+E))，辅助O(V)。邻接表的右点编号为0..right_size−1；匹配数组−1表示未匹配。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

adj=[[0,1],[0],[1,2]]; size,match=maximum_bipartite_matching(adj,3)
show_table(['左点','候选右点','最终匹配'],[(u,a,match[u]) for u,a in enumerate(adj)])

左点,候选右点,最终匹配
0,"[0, 1]",1
1,[0],0
2,"[1, 2]",2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert not is_bipartite([[1,2],[0,2],[0,1]])
assert is_bipartite([[1,3],[0,2],[1,3],[0,2]])
assert maximum_bipartite_matching([[0,1],[0]],2)[0]==2
from itertools import product

def brute(adj,m,u=0,used=0):
    if u==len(adj): return 0
    return max([brute(adj,m,u+1,used)]+[1+brute(adj,m,u+1,used|(1<<v)) for v in adj[u] if not used>>v&1])
for bits in product(range(2),repeat=9):
    adj=[[v for v in range(3) if bits[u*3+v]] for u in range(3)]
    size,match=maximum_bipartite_matching(adj,3)
    assert size==brute(adj,3)
    assert len({v for v in match if v!=-1})==size
    assert all(v==-1 or v in adj[u] for u,v in enumerate(match))
print('N087: 所有本章断言通过')

N087: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把考场冲突图按列奇偶划分。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较最大匹配与贪心随便配对。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 785. Is Graph Bipartite?（canonical）
- 886. Possible Bipartition（canonical）
- 1349. Maximum Students Taking Exam（extension）：[官方入口](https://leetcode.com/problems/maximum-students-taking-exam/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。